In [28]:
import torch
import pandas as pd
pd.set_option('display.width', 1000)

In [29]:
dim = 512
r = 8
n = 10

W = torch.randn(dim, dim)
A0 = torch.randn(dim, r)
A1 = torch.randn(dim, r)
B0 = torch.randn(r, dim)
B1 = torch.randn(r, dim)



X = torch.randn(dim, n)

In [30]:
from nnt.profiling.torch_profiler import TorchProfiler


with TorchProfiler() as prof:
    with prof.record_context("naive"):
        H = W @ X + A0 @ (B0 @ X) + A1 @ (B1 @ X)
    with prof.record_context("soups"):
        scal0 = torch.randn(1)
        scal1 = torch.randn(1)
        H = W @ X + A0 @ (scal0 * (B0 @ X)) + A1 @ (scal1 * (B1 @ X))
    with prof.record_context("vera"):
        veraA0 = torch.randn(dim)
        veraB0 = torch.randn(r)
        veraA1 = torch.randn(dim)
        veraB1 = torch.randn(r)
        H = W @ X
        H += (veraA0 * (A0 @ (veraB0 * (B0 @ X).T).T).T).T
        H += (veraA1 * (A1 @ (veraB1 * (B1 @ X).T).T).T).T
    with prof.record_context("veraB"):
        veraA0 = torch.randn(dim)
        veraB0 = torch.randn(r)
        veraA1 = torch.randn(dim)
        veraB1 = torch.randn(r)
        H = W @ X
        H += (A0 @ (veraB0 * (B0 @ X).T).T)
        H += (A1 @ (veraB1 * (B1 @ X).T).T)
print(prof.get_flops_by_step())
df = prof.to_pandas()

             flops
__init__         0
naive      5580800
__other__        0
soups      5580802
vera       5571600
veraB      5570576


/home/vince/Development/python/nn_trainer/.venv/lib/python3.12/site-packages/torch/autograd/profiler.py:267: UserWarning: CUDA is not available, disabling CUDA profiling
  warn("CUDA is not available, disabling CUDA profiling")


In [31]:
df = df[["flops", "name", "record_step", "input_shapes"]]
df = df[df["flops"] != 0]
def set_hint(row):
    if str(row["input_shapes"]) == "[[512, 512], [512, 10]]":
        return "ptw * input"
    if str(row["input_shapes"]) == "[[8, 512], [512, 10]]":
        return "B * input"
    if str(row["input_shapes"]) == "[[512, 8], [8, 10]]":
        return "A * intermediate"
    if row["name"] == "aten::add":
        return "add"
    return ""
df["hint"] = df.apply(set_hint, axis=1)

for rc in df["record_step"].unique():
    rc_df = df[df["record_step"] == rc]
    print(f"Step {rc}")
    print(rc_df)

Step naive
      flops       name record_step                input_shapes              hint
2   5242880   aten::mm       naive     [[512, 512], [512, 10]]       ptw * input
7     81920   aten::mm       naive       [[8, 512], [512, 10]]         B * input
12    81920   aten::mm       naive         [[512, 8], [8, 10]]  A * intermediate
16     5120  aten::add       naive  [[512, 10], [512, 10], []]               add
18    81920   aten::mm       naive       [[8, 512], [512, 10]]         B * input
23    81920   aten::mm       naive         [[512, 8], [8, 10]]  A * intermediate
27     5120  aten::add       naive  [[512, 10], [512, 10], []]               add
Step soups
      flops       name record_step                input_shapes              hint
35  5242880   aten::mm       soups     [[512, 512], [512, 10]]       ptw * input
40    81920   aten::mm       soups       [[8, 512], [512, 10]]         B * input
44        1  aten::mul       soups              [[1], [8, 10]]                  
46    